In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("final_merged(단칼)_v1.xlsx")

> 파생변수 1. is_weekend  

WATCH_DAY: 정수 → datetime 변환  
월=0, 화=1, ..., 토=5, 일=6  
토일이면 1, 평일이면 0  

In [3]:
df["WATCH_DAY"] = pd.to_datetime(
	df["WATCH_DAY"], format="%Y%m%d"
)
df['weekday'] = df['WATCH_DAY'].dt.dayofweek
df['is_weekend'] = (df['weekday'] >= 5).astype(int)

# is_weekend를 WATCH_DAY 바로 뒤로 이동
cols = df.columns.tolist()
cols.remove('is_weekend')
watch_day_idx = cols.index('WATCH_DAY')
cols.insert(watch_day_idx + 1, 'is_weekend')
df = df[cols]

# 중간 계산용 컬럼 제거
df = df.drop(columns=['weekday'])

> 파생변수 2. days_since_reg / days_before_end

`days_since_reg` = WATCH_DAY - reg_date (구독 시작 후 며칠째 시청했나)  
`days_before_end` = end_date - WATCH_DAY (만료까지 며칠 남았을 때 시청했나)

---

**[예시 세션] 유저 A의 시청 기록** (reg_date: 3/1, end_date: 4/1, duration_days: 31)

| WATCH_DAY | days_since_reg | days_before_end |
|---|---|---|
| 3/1 | 0 | 31 |
| 3/10 | 9 | 22 |
| 3/20 | 19 | 12 |
| 3/25 | 24 | 7 |

**[유저 단위] 집계 후**

| 집계 피처 | 값 | 해석 |
|---|---|---|
| `days_since_reg_max` | 24 | 구독 시작 24일째가 마지막 시청 |
| `days_before_end_min` | 7 | 마지막 시청이 만료 **7일 전**에 끊겼다 |

**왜 이탈 예측에 유효한가?**

| | `days_before_end_min` |
|---|---|
| 유저 A (이탈) | 7 |
| 유저 B (재구독) | 1 |

유저 B는 만료 하루 전까지 봤고, 유저 A는 7일 전에 이미 끊었다.  
"구독을 마지막까지 알뜰하게 썼냐"를 숫자로 표현하는 피처.

In [ ]:
df["days_since_reg"] = (df["WATCH_DAY"] - df["reg_date"]).dt.days
df["days_before_end"] = (df["end_date"] - df["WATCH_DAY"]).dt.days

# 두 컬럼을 is_weekend 바로 뒤로 이동
cols = df.columns.tolist()
for col in ["days_since_reg", "days_before_end"]:
    cols.remove(col)
is_weekend_idx = cols.index("is_weekend")
cols.insert(is_weekend_idx + 1, "days_since_reg")
cols.insert(is_weekend_idx + 2, "days_before_end")
df = df[cols]

> 파생변수 3. is_rewatch / watch_gap

| 파생변수 | 계산 | 의미 |
|---|---|---|
| `is_rewatch` | 같은 유저 + 같은 MOVIE_NUM 반복 시청 여부 | 이 세션이 재시청이면 1 |
| `watch_gap` | 같은 유저의 직전 시청일과의 간격 (일) | 며칠 만에 다시 시청했나 |

유저 단위 집계 시:
- `is_rewatch`의 mean → `rewatch_ratio` (전체 세션 중 재시청 비율)
- `watch_gap`의 max/mean/std → 시청 공백 최대값, 평균, 규칙성

In [ ]:
# 정렬 (is_rewatch, watch_gap 계산을 위해 순서 중요)
df = df.sort_values(['USER_KEY', 'WATCH_DAY', 'WATCH_SEQ']).reset_index(drop=True)

# is_rewatch: 같은 유저가 같은 영화를 이전에 시청한 적 있으면 1
df['is_rewatch'] = (df.groupby(['USER_KEY', 'MOVIE_NUM']).cumcount() > 0).astype(int)

# watch_gap: 같은 유저의 직전 시청일과의 간격 (일), 첫 시청은 NaN
df['watch_gap'] = df.groupby('USER_KEY')['WATCH_DAY'].transform(lambda x: x.diff().dt.days)

> 유저 단위 집계

세션 단위 데이터를 `USER_KEY` 기준으로 집계하여 1행 = 1유저 형태로 변환.

| 집계 피처 | 설명 |
|---|---|
| `total_sessions` | 총 시청 세션 수 |
| `active_days` | 실제 시청한 고유 날짜 수 |
| `activity_rate` | active_days / duration_days (구독 기간 활용도) |
| `max_daily_sessions` | 하루 최대 시청 편수 |
| `weekend_watch_ratio` | 주말 시청 비율 |
| `first_watch_day` | 가입 후 첫 시청까지 며칠 걸렸나 |
| `last_watch_days_since_reg` | 마지막 시청이 가입 후 며칠째 |
| `days_before_end_min` | 마지막 시청이 만료 며칠 전 |
| `unique_movies` | 시청한 고유 영화 수 |
| `rewatch_ratio` | 재시청 비율 |
| `total_duration` | 총 시청 시간 (분) |
| `avg_duration` | 세션당 평균 시청 시간 (분) |
| `watch_gap_max` | 시청 간 최대 공백 (일) |
| `watch_gap_mean` | 시청 간 평균 공백 (일) |
| `watch_gap_std` | 시청 간격의 표준편차 (규칙성, 낮을수록 규칙적) |
| `binge_day_count` | 하루 3편 이상 시청한 날 수 |
| `price_per_day` | amount / duration_days (하루 구독료 체감 단가) |

In [ ]:
dur_col = '(파생)duration_days'

# binge_day_count: 하루에 3편 이상 본 날 수
binge = (
    df.groupby(['USER_KEY', 'WATCH_DAY'])['WATCH_SEQ'].max()
    .reset_index()
    .query('WATCH_SEQ >= 3')
    .groupby('USER_KEY')
    .size()
    .rename('binge_day_count')
)

user_df = df.groupby('USER_KEY').agg(
    USER_NUM=('USER_NUM', 'first'),
    product_cd=('product_cd', 'first'),
    amount=('amount', 'first'),
    billing_method=('billing_method', 'first'),
    concurrent_streams=('concurrent_streams', 'first'),
    promotion_yn=('promotion_yn', 'first'),
    is_churn_prevented=('is_churn_prevented', 'first'),
    repurchase=('repurchase', 'first'),
    payment_device=('payment_device', 'first'),
    is_user_verified=('is_user_verified', 'first'),
    gender=('gender', 'first'),
    age=('age', 'first'),
    reg_date=('reg_date', 'first'),
    reg_hour=('reg_hour', 'first'),
    end_date=('end_date', 'first'),
    duration_days=(dur_col, 'first'),
    # 시청 행동
    total_sessions=('WATCH_SEQ', 'count'),
    active_days=('WATCH_DAY', 'nunique'),
    max_daily_sessions=('WATCH_SEQ', 'max'),
    weekend_watch_ratio=('is_weekend', 'mean'),
    first_watch_day=('days_since_reg', 'min'),
    last_watch_days_since_reg=('days_since_reg', 'max'),
    days_before_end_min=('days_before_end', 'min'),
    # 콘텐츠 소비
    unique_movies=('MOVIE_NUM', 'nunique'),
    rewatch_ratio=('is_rewatch', 'mean'),
    total_duration=('DURATION', 'sum'),
    avg_duration=('DURATION', 'mean'),
    # 시청 간격
    watch_gap_max=('watch_gap', 'max'),
    watch_gap_mean=('watch_gap', 'mean'),
    watch_gap_std=('watch_gap', 'std'),
).reset_index()

# binge_day_count 합류
user_df = user_df.merge(binge, on='USER_KEY', how='left')
user_df['binge_day_count'] = user_df['binge_day_count'].fillna(0).astype(int)

# 집계 후 파생
user_df['activity_rate'] = user_df['active_days'] / user_df['duration_days']
user_df['price_per_day'] = user_df['amount'] / user_df['duration_days']
user_df['watch_gap_std'] = user_df['watch_gap_std'].fillna(0)

output_path = 'final_merged_user(단칼)_v2.xlsx'
user_df.to_excel(output_path, index=False)
print(f"저장 완료: {output_path}")
print(f"Shape: {user_df.shape}")
print(f"컬럼:\n{user_df.columns.tolist()}")

In [ ]:
# # 엑셀로 저장
# output_path = "final_merged(단칼)_v2.xlsx"
# df.to_excel(output_path, index=False)

# print(f"저장 완료: {output_path}")
# print(df.columns[:6].tolist())

저장 완료: final_merged(단칼)_v2.xlsx
['USER_KEY', 'USER_NUM', 'WATCH_DAY', 'is_weekend', 'WATCH_SEQ', 'product_cd']
